In [80]:
library(data.table)
library(SingleCellExperiment)
library(dplyr)
library(Matrix)


Attaching package: ‘Matrix’


The following object is masked from ‘package:S4Vectors’:

    expand




In [37]:
io = list()
io$atlas.basedir <- "/rds/project/rds-SDzz0CATGms/users/bt392/atlasses/gastrulation/pijuansala2019_gastrulation10x"
io$rna.atlas.metadata <- file.path(io$atlas.basedir,"sample_metadata.txt.gz")
io$rna.atlas.sce <- file.path(io$atlas.basedir,"processed/SingleCellExperiment.rds")
io$rna.atlas.marker_genes <- file.path(io$atlas.basedir,"results/marker_genes/all_stages/marker_genes.txt.gz")
io$transitions = 'all_transitions.csv'
io$TFs = '/rds/project/rds-SDzz0CATGms/users/bt392/atlasses/TFs.txt'

In [35]:
getwd()

[1] "/rds/project/rds-SDzz0CATGms/users/bt392/atlasses/gastrulation/pijuansala2019_gastrulation10x/results/WOT"

In [14]:
transitions = fread(io$transitions, header=TRUE)

In [4]:
meta_atlas <- fread(io$rna.atlas.metadata)

In [33]:
marker_genes = fread(io$rna.atlas.marker_genes)

In [42]:
marker_TFs = marker_genes[gene %in% stringr::str_to_title(fread(io$TFs, header=F)$V1)]

In [49]:
TFs_keep = unique(marker_TFs[order(-score), gene])

In [50]:
TFs_keep

[1] "Hoxa10"  "Twist2"  "Pitx1"   "Gsc"     "Mef2c"   "Nkx2-5"  "Tbx5"   
  [8] "Elk3"    "Klf7"    "Cebpd"   "Ascl2"   "Id2"     "Elf5"    "Zfp42"  
 [15] "Hnf4a"   "Creb3l3" "Foxo4"   "Hoxd1"   "Etv2"    "Hoxc8"   "Hoxb9"  
 [22] "Mesp2"   "Dlx2"    "Tfap2b"  "Sox10"   "Nr2f1"   "Foxj1"   "T"      
 [29] "Noto"    "Rfx3"    "Six1"    "Tcf15"   "Klf4"    "Klf5"    "Tbx6"   
 [36] "Pax6"    "Dlx5"    "Foxq1"   "Foxa3"   "Lhx1"    "Mixl1"   "Gata2"  
 [43] "Gata5"   "Fos"     "Klf2"    "Pou3f1"  "Mybl2"   "Tfeb"    "Batf3"  
 [50] "Tfap2c"  "Cebpb"   "Rhox9"   "Hmg20b"  "Hoxd9"   "Creb3l1" "Mesp1"  
 [57] "Hoxc6"   "Hoxb8"   "Hoxc4"   "Sox9"    "Tfap2a"  "Meox1"   "Foxc2"  
 [64] "Ebf1"    "Gata6"   "Rhox6"   "Gata4"   "Junb"    "Sox7"    "Meis1"  
 [71] "Nanog"   "Lef1"    "Hoxd4"   "Tbx20"   "Hoxc9"   "Hoxa9"   "Fli1"   
 [78] "Etv4"    "Cdx1"    "Nkx1-2"  "Hoxa1"   "Ets2"    "Atf3"    "Klf1"   
 [85] "Nfe2"    "Klf9"    "Hnf1b"   "Foxa1"   "Osr1"    "Hand1"   "Cdx4"   
 [92] "Hoxa7"   "Tcf12"   "Foxa2"   "Sp5"     "Xbp1"    "Sox17"   "Snai1"  
 [99] "Hoxb6"   "Twist1"  "Prrx2"   "Eomes"   "Otx2"    "Runx1"   "Gata1"  
[106] "Hes7"    "Ets1"    "Lyl1"    "Esrra"   "Pitx2"   "Pax3"    "Tcf7l2" 
[113] "Alx1"    "Hoxb1"   "Ovol2"   "Hand2"   "Bcl11a"  "Gfi1b"   "Hoxb7"  
[120] "Rxrg"    "Gbx2"    "Creb3"   "Foxf1"   "Hoxa3"   "Smad1"   "Irx5"   
[127] "Jund"    "Isl1"    "Tcf21"   "Tbx3"    "Bbx"     "Hhex"    "Tal1"   
[134] "Cdx2"    "Hoxb5"   "Sox2"    "Hoxb4"   "Zic3"    "Msx2"    "Id1"    
[141] "Elf3"    "Hbp1"    "Klf6"    "Hoxa2"   "Evx1"    "Foxb1"   "Jun"    
[148] "Srf"     "Nr6a1"   "Sox4"    "Irx3"    "Msx1"    "Zic5"    "Meis2"  
[155] "Pou5f1"  "Arid5b"  "Rarg"    "Tead1"   "Mycl"    "Hes1"    "Nr2f6"

In [53]:
genes = unique(c(TFs_keep, unique(marker_genes[order(-score), gene])))

In [58]:
load_SingleCellExperiment <- function(file, normalise = FALSE, features = NULL, cells = NULL, remove_non_expressed_genes = FALSE) {
  library(SingleCellExperiment); library(scran); library(scater);
  sce <- readRDS(file)
  if (!is.null(cells)) sce <- sce[,cells]
  if (!is.null(features)) sce <- sce[features,]
  if (remove_non_expressed_genes) sce <- sce[which(Matrix::rowSums(counts(sce))>15),]
  if (normalise) sce <- logNormCounts(sce)
  return(sce)
}

sce_atlas <- load_SingleCellExperiment(io$rna.atlas.sce, normalise = TRUE, remove_non_expressed_genes = TRUE)

In [62]:
genes_ens_id = marker_genes[match(genes, gene), ens_id]

In [100]:
sce_atlas_markers = sce_atlas[genes_ens_id,]
rownames(sce_atlas_markers) = genes

In [101]:
logcounts = round(logcounts(sce_atlas_markers), 2)

In [116]:
minmax = function(x){
    (x-min(x))/(max(x) - min(x))
}

In [135]:
logcounts_minmax = t(apply(logcounts, 1, minmax))
logcounts_minmax = as(logcounts_minmax, 'sparseMatrix')

Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 1.8 GiB”


In [136]:
TFs_matrix = logcounts[TFs_keep,transitions[[1]]]

In [137]:
writeMM(TFs_matrix, 'input_matrix.mtx')

NULL

In [138]:
full_matrix = logcounts[,transitions[[2]]]

In [139]:
summary(as.vector(full_matrix))

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
 0.0000  0.0000  0.0000  0.4208  0.3900 11.4800 

In [140]:
writeMM(full_matrix, 'output_matrix.mtx')

NULL

In [143]:
meta_predictions = meta_atlas[match(transitions[[2]], cell),]

In [144]:
fwrite(meta_predictions, 'meta_predictions.csv')

In [171]:
unique(marker_genes$celltype)

[1] "Allantois"                      "Anterior_Primitive_Streak"     
 [3] "Blood_progenitors_1"            "Blood_progenitors_2"           
 [5] "Cardiomyocytes"                 "Caudal_Mesoderm"               
 [7] "Caudal_epiblast"                "Caudal_neurectoderm"           
 [9] "Def._endoderm"                  "Endothelium"                   
[11] "Epiblast"                       "Erythroid1"                    
[13] "Erythroid2"                     "Erythroid3"                    
[15] "ExE_ectoderm"                   "ExE_endoderm"                  
[17] "ExE_mesoderm"                   "Forebrain_Midbrain_Hindbrain"  
[19] "Gut"                            "Haematoendothelial_progenitors"
[21] "Intermediate_mesoderm"          "Mesenchyme"                    
[23] "Mixed_mesoderm"                 "NMP"                           
[25] "Nascent_mesoderm"               "Neural_crest"                  
[27] "Notochord"                      "PGC"                           
[29] "Paraxial_mesoderm"              "Parietal_endoderm"             
[31] "Pharyngeal_mesoderm"            "Primitive_Streak"              
[33] "Rostral_neurectoderm"           "Somitic_mesoderm"              
[35] "Spinal_cord"                    "Surface_ectoderm"              
[37] "Visceral_endoderm"

In [188]:
head(marker_genes[celltype=='Endothelium'], 7)

celltype,gene,ens_id,score,N
<chr>,<chr>,<chr>,<dbl>,<int>
Endothelium,Acvrl1,ENSMUSG00000000530,1,1
Endothelium,Icam2,ENSMUSG00000001029,1,1
Endothelium,N4bp3,ENSMUSG00000001053,1,1
Endothelium,Ramp2,ENSMUSG00000001240,1,4
Endothelium,Col18a1,ENSMUSG00000001435,1,1
Endothelium,Tubb6,ENSMUSG00000001473,1,3
Endothelium,Esam,ENSMUSG00000001946,1,2


In [194]:
grep('Tubb6', genes)-1

[1] 241

In [145]:
genes

[1] "Hoxa10"        "Twist2"        "Pitx1"         "Gsc"          
   [5] "Mef2c"         "Nkx2-5"        "Tbx5"          "Elk3"         
   [9] "Klf7"          "Cebpd"         "Ascl2"         "Id2"          
  [13] "Elf5"          "Zfp42"         "Hnf4a"         "Creb3l3"      
  [17] "Foxo4"         "Hoxd1"         "Etv2"          "Hoxc8"        
  [21] "Hoxb9"         "Mesp2"         "Dlx2"          "Tfap2b"       
  [25] "Sox10"         "Nr2f1"         "Foxj1"         "T"            
  [29] "Noto"          "Rfx3"          "Six1"          "Tcf15"        
  [33] "Klf4"          "Klf5"          "Tbx6"          "Pax6"         
  [37] "Dlx5"          "Foxq1"         "Foxa3"         "Lhx1"         
  [41] "Mixl1"         "Gata2"         "Gata5"         "Fos"          
  [45] "Klf2"          "Pou3f1"        "Mybl2"         "Tfeb"         
  [49] "Batf3"         "Tfap2c"        "Cebpb"         "Rhox9"        
  [53] "Hmg20b"        "Hoxd9"         "Creb3l1"       "Mesp1"        
  [57] "Hoxc6"         "Hoxb8"         "Hoxc4"         "Sox9"         
  [61] "Tfap2a"        "Meox1"         "Foxc2"         "Ebf1"         
  [65] "Gata6"         "Rhox6"         "Gata4"         "Junb"         
  [69] "Sox7"          "Meis1"         "Nanog"         "Lef1"         
  [73] "Hoxd4"         "Tbx20"         "Hoxc9"         "Hoxa9"        
  [77] "Fli1"          "Etv4"          "Cdx1"          "Nkx1-2"       
  [81] "Hoxa1"         "Ets2"          "Atf3"          "Klf1"         
  [85] "Nfe2"          "Klf9"          "Hnf1b"         "Foxa1"        
  [89] "Osr1"          "Hand1"         "Cdx4"          "Hoxa7"        
  [93] "Tcf12"         "Foxa2"         "Sp5"           "Xbp1"         
  [97] "Sox17"         "Snai1"         "Hoxb6"         "Twist1"       
 [101] "Prrx2"         "Eomes"         "Otx2"          "Runx1"        
 [105] "Gata1"         "Hes7"          "Ets1"          "Lyl1"         
 [109] "Esrra"         "Pitx2"         "Pax3"          "Tcf7l2"       
 [113] "Alx1"          "Hoxb1"         "Ovol2"         "Hand2"        
 [117] "Bcl11a"        "Gfi1b"         "Hoxb7"         "Rxrg"         
 [121] "Gbx2"          "Creb3"         "Foxf1"         "Hoxa3"        
 [125] "Smad1"         "Irx5"          "Jund"          "Isl1"         
 [129] "Tcf21"         "Tbx3"          "Bbx"           "Hhex"         
 [133] "Tal1"          "Cdx2"          "Hoxb5"         "Sox2"         
 [137] "Hoxb4"         "Zic3"          "Msx2"          "Id1"          
 [141] "Elf3"          "Hbp1"          "Klf6"          "Hoxa2"        
 [145] "Evx1"          "Foxb1"         "Jun"           "Srf"          
 [149] "Nr6a1"         "Sox4"          "Irx3"          "Msx1"         
 [153] "Zic5"          "Meis2"         "Pou5f1"        "Arid5b"       
 [157] "Rarg"          "Tead1"         "Mycl"          "Hes1"         
 [161] "Nr2f6"         "Sgce"          "Pgf"           "Cox4i2"       
 [165] "Slc38a4"       "Suv39h1"       "Tmem119"       "Plac1"        
 [169] "Cnn1"          "Mybpc3"        "Hspb7"         "Crip1"        
 [173] "Wnt2"          "Dstn"          "Mpped2"        "Unc45b"       
 [177] "Gyg"           "Csrp2"         "Myl7"          "Pgam2"        
 [181] "Vamp2"         "Asb2"          "Ryr2"          "Cap2"         
 [185] "Pdlim7"        "Thbs4"         "Fitm1"         "Col2a1"       
 [189] "Popdc2"        "Ankrd1"        "Homer2"        "Des"          
 [193] "Tnnt2"         "Tnni1"         "Pdlim5"        "Smarcd3"      
 [197] "Atp2a2"        "Csrp3"         "Tgfb1i1"       "Mylk3"        
 [201] "Rrad"          "Acta1"         "Tagln"         "Tpm1"         
 [205] "Tnni3"         "Acta2"         "3632451O06Rik" "Hspb2"        
 [209] "Rbm24"         "Nexn"          "Prss23"        "Ppp1r14c"     
 [213] "Sh3bgr"        "Apobec2"       "Myh6"          "Smpx"         
 [217] "Dcp1b"         "Ccdc141"       "Fbxl22"        "Ttn"          
 [221] "Actn2"         "Myh7"          "Nebl"          "Smyd1"        
 [225] "Myl3"          "Eno